In [3]:
import polars as pl
from pathlib import Path

In [4]:
DATA_GENERAL = Path("../data_general")
DATA_PERSONAL = Path("../data_personal/Spotify Extended Streaming History")
OUT_DATA = Path('../output') 

In [5]:
lazy_general_df = pl.scan_parquet(DATA_GENERAL / "spotify_audio_features_*.parquet")

In [6]:
general_df = (
    lazy_general_df
    .filter(pl.col('null_response') == 0)   
    .drop('null_response')
    .collect()
)

In [7]:
general_df = general_df.rename({ 'id' : 'spotify_track_uri' })

In [8]:
display(general_df.head())

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2Pe9cbhOTvOUTDE4bl7zzl""","""I dreamt you died""",0,630506,4,6,0,87.683,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
"""0wP732NKm8XgXu78XLRWoR""","""It's Death""",0,97216,4,5,1,105.298,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
"""22L6EJdnjx8oIo7GiF9hLe""","""Preliminary""",0,75180,4,0,1,117.657,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
"""3a519lgQ13JXNi0G73mwMT""","""Disparage""",0,149447,4,5,0,100.685,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
"""27yP7p2lxWYTtnldRN8Kzx""","""Cut Down""",0,120816,4,7,1,123.499,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


In [9]:
personal_data_frames = {}
for num in range (2022,2027) :
    personal_data_frames[num] = pl.read_json(
        DATA_PERSONAL / f'Streaming_History_Audio_{num}.json',
        infer_schema_length=None  
        )

In [23]:
song_uri = '39tEggsuFc41pIFmajsWb8'

In [24]:
stats = general_df.filter(pl.col('spotify_track_uri') == song_uri)

In [25]:
display(stats)

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""39tEggsuFc41pIFmajsWb8""","""Песня красноармейца""",34,154587,4,0,0,98.503,0.388,0.851,-6.026,0.04,0.0000477,0.199,0.29,0.724


# I have priority on the  energy, instrumentalness and valence

In [28]:
cof_tempo = 0.1
cof_dance = 0.2
cof_energy = 0.1 
cof_loud = None
cof_speech = None
cof_acoustic = None
cof_instr = 0.2 
cof_live = 0.05
cof_valence = 0.1

features = [ 
    (cof_tempo, 'tempo') ,       
    ( cof_dance, 'danceability' ),     
    (cof_energy, 'energy'),    
    (cof_loud, 'loudness' ),    
    (cof_speech,  'speechiness' ),   
    (cof_acoustic,'acousticness') ,
    (cof_instr, 'instrumentalness'),    
    (cof_live, 'liveness'),    
    (cof_valence, 'valence'),     
]

In [29]:
conditions = []
for cof, state_key in features:
    if cof:
        state = stats[state_key]
        cond = pl.col(state_key).is_between(
            state * (1 - cof),
            state * (1 + cof)
        )
        conditions.append(cond)


In [30]:
if conditions:
    result = (
        general_df.lazy()
        .filter(conditions)
        .collect()
    )

In [31]:
print(len(result))
result = result.with_columns(
    pl.format("spotify:track:{}", pl.col("spotify_track_uri")).alias("spotify_track_uri")
)
display(result)


106


spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""spotify:track:3SaLm6z51350Oy4S…","""Imposter Syndrome""",0,203408,4,6,1,102.023,0.325,0.907,-7.997,0.102,0.000558,0.18,0.297,0.751
"""spotify:track:6zP3mlOLoi4uYIbS…","""Passion reverse""",0,305862,3,7,1,96.943,0.404,0.929,-7.069,0.078,0.00361,0.163,0.294,0.693
"""spotify:track:5h75tNQtnW6czUIL…","""Aag Budh Ko Chhod De""",0,917813,4,11,0,96.659,0.455,0.855,-4.965,0.116,0.69,0.202,0.278,0.792
"""spotify:track:0n1hMwF8Ld590iQY…","""Like Love""",0,146256,4,9,1,92.495,0.388,0.931,-11.913,0.0434,0.000004,0.185,0.285,0.668
"""spotify:track:2l5GAoAQnKxINoRI…","""We Never Wanted This""",0,173507,4,2,1,91.137,0.366,0.927,-3.738,0.0432,0.000118,0.18,0.294,0.654
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""spotify:track:5UsoPZRJ5P1ZrQty…","""Morning Train - Single Version""",0,199827,4,2,1,90.735,0.464,0.88,-6.493,0.124,0.00337,0.197,0.286,0.764
"""spotify:track:05gPLrJRmXDtpP1x…","""Desorden Social""",0,150576,4,10,1,104.308,0.325,0.869,-10.144,0.0886,0.346,0.183,0.299,0.656
"""spotify:track:05iZHIpyCGalKbJ1…","""The Hollywood Scene""",2,148267,4,4,0,105.899,0.436,0.867,-7.303,0.0779,0.435,0.179,0.279,0.686


In [32]:
file_path = OUT_DATA / f"Like '{stats['name'][0]}'.csv"
result.select('spotify_track_uri').write_csv(file_path , separator=',')